## The problem Spark solves

Imagine a CSV with two hundred gigabytes of bank transactions and a laptop with sixteen gigabytes of RAM. Pandas crashes the moment you try to load it. You could split the file into chunks and loop, but you now own a small distributed system: which chunks finished, which failed, how do you do a group-by that spans the whole file?

Spark exists because this pattern — data too big for one machine, work that needs to be coordinated across many — keeps appearing past a few hundred gigabytes. Instead of writing your own chunking and coordination layer, Spark gives you one engine that handles it, behind a familiar table-shaped API.

## What Spark actually is

Apache Spark is a **distributed compute engine**. You write code as if you have one giant table. Spark slices the data into pieces called **partitions**, ships those pieces to many machines, runs your code on each in parallel, and brings the result back.

The mental shift from pandas: pandas runs your code in one process, on one machine, against memory it owns directly. Spark runs a *plan* you describe — across many processes, on many machines, against memory those processes own. You never move data between machines by hand; Spark does it for you.

## The cluster — driver, executors, cluster manager

Three roles whenever Spark runs:

- **Driver** — the process running your Python script. Owns the SparkSession, builds the execution plan, decides what work to send out. The project manager — never lifts a box itself.
- **Executors** — worker processes that hold partitions in memory and run the actual computations. The delivery drivers, each handling their own route.
- **Cluster manager** — allocates machines to your Spark application. It does not understand your job; it just hands out resources. Options include Standalone, YARN, Kubernetes, or **local mode** (driver and executors collapsed into one JVM on your laptop).

![Spark Cluster Architecture](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-cluster-architecture.svg)

## Memory-first execution

The big leap over the previous generation, Hadoop MapReduce, is one design choice: keep intermediate results in executor RAM instead of writing them to disk between every step. A multi-step pipeline used to do a full disk round-trip per step. In Spark, data stays in memory until something forces it out.

That single decision is most of Spark's speed advantage. Caching, broadcast joins, and the rest are details on top of it — covered in notebook 07.

## From your code to distributed work

When you write `df.filter(...).groupBy(...).count()`, here is what happens behind the scenes:

1. The driver builds a **logical plan** from your DataFrame chain.
2. The **Catalyst** optimizer rewrites it — pushing filters down, reordering joins, dropping unused columns.
3. The plan becomes a **physical plan**, then a **DAG** of stages.
4. Each stage splits into **tasks** — one task per partition.
5. Tasks ship to executors and run in parallel across their CPU cores.

Three exam-vocabulary words to lock in now:

- **Job** — what one action triggers. One `.count()` call → one job.
- **Stage** — a slice of work between shuffles. A new shuffle starts a new stage.
- **Task** — the smallest unit of work. One partition, processed by one executor core.

![Job, Stages, Tasks](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-job-stages-tasks.svg)

## Deployment modes

How the driver and executors actually get placed onto machines:

- **`local[*]`** — everything in one JVM on your laptop. The mode this notebook uses. The `*` means "use all available cores."
- **Standalone** — Spark's built-in cluster manager. Simple, no external dependency.
- **YARN** — runs on Hadoop clusters. Common in legacy on-prem stacks.
- **Kubernetes** — driver and executors as pods. The modern cloud-native default.
- **Databricks** — managed Spark; the cluster manager is hidden behind the platform.

For learning, `local[*]` is plenty. For the exam, you should at minimum recognize each name and what it runs on.

![Spark Setup Options](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-setup-options.svg)

## Setup — your first SparkSession

Local install (Java 17+ must already be present):

```bash
pip install pyspark==3.5.3 delta-spark==3.2.1
```

This repo standardizes on `configure_spark_with_delta_pip` so the Delta JARs are loaded into the JVM at session start. Every notebook in the series can use Delta without separate setup.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("SparkFoundations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.version

## Lazy evaluation — first taste

Spark operations split into two kinds:

- **Transformations** — `filter`, `select`, `groupBy`, `join`. They build the recipe. Nothing runs.
- **Actions** — `count`, `show`, `collect`, `write`. They trigger execution against the recipe.

Why this matters: by the time you call an action, Spark sees the *entire* chain of transformations and can plan the most efficient execution. If transformations ran eagerly, each step would commit before the next, and the optimizer would have nothing to optimize.

![Lazy Evaluation](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-lazy-evaluation.svg)

In [ ]:
# A tiny in-memory DataFrame of bank customers — enough to demonstrate Spark mechanics.
data = [
    ("CUST0001", "Aarav Sharma",  "Mumbai",    780),
    ("CUST0002", "Priya Nair",    "Delhi",     650),
    ("CUST0003", "Rohan Gupta",   "Bengaluru", 720),
    ("CUST0004", "Anjali Mehta",  "Pune",      810),
    ("CUST0005", "Vikram Reddy",  "Hyderabad", 590),
]
columns = ["customer_id", "full_name", "city", "credit_score"]
df = spark.createDataFrame(data, columns)

# Chain two transformations. Notice: no output, no work done.
high_credit = df.filter(df.credit_score >= 700).select("customer_id", "full_name", "credit_score")
print("Transformations defined. Nothing has executed yet.")

In [ ]:
# Inspect the plan Spark built up. Catalyst will already have pushed the filter close to the scan.
high_credit.explain()

In [ ]:
# .show() is an action — this is the call that actually triggers execution.
high_credit.show()

## Hello Spark — a multi-step query

The cells above already proved the install: a SparkSession started, a DataFrame was built, an action ran. One more pass to confirm a multi-step query works end to end — same five customers, this time grouped through SQL.

In [ ]:
# Register the DataFrame as a temporary SQL view, then query it.
df.createOrReplaceTempView("customers")

spark.sql("""
    SELECT city,
           COUNT(*)                    AS num_customers,
           ROUND(AVG(credit_score), 0) AS avg_credit_score
    FROM customers
    GROUP BY city
    ORDER BY avg_credit_score DESC
""").show()

## When NOT to use Spark

Spark is the wrong tool when:

- **Your data fits comfortably in pandas.** Below roughly ten gigabytes on a modern laptop, pandas is faster, simpler, and has no cluster overhead.
- **You need single-row, sub-millisecond lookups.** Spark is throughput-optimized, not latency-optimized. Reach for Postgres, DynamoDB, or Redis.
- **You need OLTP semantics.** Spark is for analytical workloads — read-heavy, write-batched. It is not a transactional database.
- **You only need a scheduled script.** A `cron` plus a Python script can be the right answer; reach for Spark when scale or fault tolerance actually demand it.

The honest rule: Spark earns its complexity when single-machine tools stop working.

In [ ]:
spark.stop()